# Parquet with Python: A Low-Level Introduction

This notebook uses **PyArrow directly**—not pandas—to create, read, and inspect a Parquet file. It focuses on the format's building blocks: schema, row groups, column chunks, pages, encodings, compression, statistics, the footer, and magic bytes.

## Common Python options

| Tool | Typical production use |
|---|---|
| **PyArrow** | Standard Python interface to Apache Arrow and Parquet; strong schema and metadata support |
| **fastparquet** | Alternative Parquet implementation, commonly used through pandas |
| **DuckDB** | Analytical SQL engine with excellent Parquet scanning and pushdown support |
| **Polars** | High-performance DataFrame engine built on Arrow-style columnar processing |

This notebook uses PyArrow because it provides a direct, well-supported Parquet API without requiring a DataFrame library.

## Installation

Run this in a terminal if PyArrow is not installed:

```powershell
python -m pip install pyarrow
```

The examples write to `C:\data\output`.


## 1. Imports and output location

`pyarrow.Table` is an in-memory columnar table. `pyarrow.parquet` reads and writes the Parquet file format.


In [ ]:
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq

OUTPUT_DIR = Path(r"C:\data\output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_PATH = OUTPUT_DIR / "events_intro.parquet"

print(f"PyArrow version: {pa.__version__}")
print(f"Output file: {PARQUET_PATH}")


## 2. Create typed, columnar data

The schema is explicit. Repeated strings in `region` and `event_type` make those columns good candidates for dictionary encoding.


In [ ]:
schema = pa.schema([
    pa.field("event_id", pa.int64(), nullable=False),
    pa.field("region", pa.string(), nullable=False),
    pa.field("event_type", pa.string(), nullable=False),
    pa.field("amount", pa.float64(), nullable=False),
    pa.field("is_priority", pa.bool_(), nullable=False),
])

row_count = 24
table = pa.Table.from_arrays(
    [
        pa.array(range(1, row_count + 1), type=pa.int64()),
        pa.array((["APAC", "EMEA", "APAC", "AMER"] * 6), type=pa.string()),
        pa.array((["view", "view", "purchase", "view", "click", "view"] * 4), type=pa.string()),
        pa.array([float(value * 10 + 0.5) for value in range(row_count)], type=pa.float64()),
        pa.array(([False, False, True, False, False, True] * 4), type=pa.bool_()),
    ],
    schema=schema,
)

print(table.schema)
print(f"Rows: {table.num_rows}; columns: {table.num_columns}")
print(table.slice(0, 5))


## 3. Write a Parquet file

- `row_group_size=8` creates three row groups, making the physical hierarchy easy to inspect.
- `compression="zstd"` compresses each data page independently.
- Dictionary encoding is requested for the two repeated string columns.
- `data_page_size` is a target, not a guarantee. Writers decide page boundaries based on encoded data and implementation details.
- Statistics support row-group and page-level filtering by capable readers.


In [ ]:
# row_group_size=100000
pq.write_table(
    table,
    PARQUET_PATH,
    row_group_size=8,
    compression="zstd",
    use_dictionary=["region", "event_type"],
    write_statistics=True,
    data_page_size=1024,
    data_page_version="1.0",
)

print(f"Wrote {PARQUET_PATH.stat().st_size:,} bytes")


## 4. Read without pandas

`pq.read_table` reconstructs an Arrow table. A reader can request only selected columns, which is one of the main benefits of columnar storage.


In [ ]:
selected = pq.read_table(PARQUET_PATH, columns=["region", "amount"])

print(selected.schema)
print(selected.slice(0, 6))

round_trip = pq.read_table(PARQUET_PATH)
assert round_trip.equals(table)
print("Round-trip validation passed.")


## 5. Inspect the file, schema, and row groups

A Parquet file contains one or more row groups. Each row group contains one column chunk per column. Each column chunk contains one or more pages.

PyArrow exposes footer and column-chunk metadata. Individual page headers are consumed internally by the reader and are not exposed as a stable, high-level Python object.


In [ ]:
parquet_file = pq.ParquetFile(PARQUET_PATH)
metadata = parquet_file.metadata

print(f"Created by: {metadata.created_by}")
print(f"Format version: {metadata.format_version}")
print(f"Rows: {metadata.num_rows}")
print(f"Row groups: {metadata.num_row_groups}")
print(f"Columns: {metadata.num_columns}")
print("\nArrow schema reconstructed from Parquet:")
print(parquet_file.schema_arrow)

for row_group_index in range(metadata.num_row_groups):
    row_group = metadata.row_group(row_group_index)
    print(
        f"Row group {row_group_index}: "
        f"rows={row_group.num_rows}, "
        f"total_byte_size={row_group.total_byte_size}"
    )


## 6. Inspect column chunks

The footer stores metadata for every column chunk, including its physical type, compression codec, encodings, statistics, sizes, and file offsets.

Encoding names commonly include:

- `PLAIN`: basic representation of values or dictionary entries
- `RLE_DICTIONARY`: data values are stored as dictionary indexes
- `RLE`: run-length/bit-packed encoding, also used for repetition and definition levels


In [ ]:
for row_group_index in range(metadata.num_row_groups):
    row_group = metadata.row_group(row_group_index)
    print(f"\nROW GROUP {row_group_index}")
    for column_index in range(row_group.num_columns):
        column = row_group.column(column_index)
        stats = column.statistics
        print(
            f"  {column.path_in_schema:<12} "
            f"type={str(column.physical_type):<10} "
            f"codec={str(column.compression):<6} "
            f"encodings={column.encodings}"
        )
        print(
            f"    values={column.num_values}, "
            f"compressed={column.total_compressed_size}, "
            f"uncompressed={column.total_uncompressed_size}, "
            f"data_page_offset={column.data_page_offset}, "
            f"dictionary_page_offset={column.dictionary_page_offset}"
        )
        if stats is not None and stats.has_min_max:
            print(f"    min={stats.min!r}, max={stats.max!r}, nulls={stats.null_count}")


## 7. Dictionary encoding: the idea

Dictionary encoding stores each distinct value once and replaces repeated values with small integer indexes.

For example:

```text
values:     APAC, EMEA, APAC, AMER, APAC
dictionary: [APAC, EMEA, AMER]
indexes:    0,    1,    0,    2,    0
```

In Parquet, the dictionary is usually stored in a dictionary page. Data pages then contain dictionary indexes, commonly represented with `RLE_DICTIONARY`. Writers may fall back to plain encoding if the dictionary grows too large or becomes ineffective.


In [ ]:
values = table.column("region").to_pylist()
dictionary = list(dict.fromkeys(values))
index_by_value = {value: index for index, value in enumerate(dictionary)}
indexes = [index_by_value[value] for value in values]

print("Dictionary:", dictionary)
print("First 12 indexes:", indexes[:12])

region_metadata = metadata.row_group(0).column(1)
print("Encodings reported by the first region column chunk:", region_metadata.encodings)
assert region_metadata.dictionary_page_offset is not None


## 8. Run-length encoding (RLE): the idea

RLE represents a repeated value as a value and a run length.

```text
input:  0, 0, 0, 1, 1, 0
runs:   (0, 3), (1, 2), (0, 1)
```

Parquet uses a hybrid **RLE/bit-packed encoding** for several integer streams. Important uses include:

- dictionary indexes
- definition levels, which indicate whether optional values are present
- repetition levels, which describe nested/repeated structures

Long runs benefit from RLE; changing sequences benefit from bit packing.


In [ ]:
def simple_rle(values):
    if not values:
        return []
    runs = []
    current = values[0]
    count = 1
    for value in values[1:]:
        if value == current:
            count += 1
        else:
            runs.append((current, count))
            current = value
            count = 1
    runs.append((current, count))
    return runs


example = [0, 0, 0, 1, 1, 0, 0, 2, 2, 2, 2]
print("Input:", example)
print("Conceptual RLE runs:", simple_rle(example))


## 9. Inspect the outer file structure

Parquet begins and ends with the four-byte magic value `PAR1`. Immediately before the trailing magic bytes is a four-byte, little-endian footer length. The footer contains Thrift-serialized file metadata.

```text
PAR1 | column data and page headers | footer metadata | footer length | PAR1
```


In [ ]:
raw = PARQUET_PATH.read_bytes()
leading_magic = raw[:4]
trailing_magic = raw[-4:]
footer_length = int.from_bytes(raw[-8:-4], byteorder="little", signed=False)
footer_start = len(raw) - 8 - footer_length

print(f"Leading magic: {leading_magic!r}")
print(f"Trailing magic: {trailing_magic!r}")
print(f"Footer length: {footer_length:,} bytes")
print(f"Footer starts at byte offset: {footer_start:,}")

assert leading_magic == b"PAR1"
assert trailing_magic == b"PAR1"
assert footer_start > 4


## 10. Read one row group

Distributed engines often schedule work at row-group granularity. Column projection can then be applied inside the selected row group.


In [ ]:
second_row_group = parquet_file.read_row_group(1, columns=["event_id", "event_type"])

print(second_row_group)
assert second_row_group.num_rows == 8


## Summary

- PyArrow can write and read Parquet directly without pandas.
- **Row groups** provide coarse-grained partitioning and parallelism.
- **Column chunks** hold one column within a row group.
- **Pages** are the main encoding and compression units inside column chunks.
- Dictionary encoding replaces repeated values with indexes.
- Parquet's hybrid RLE/bit-packed encoding efficiently stores dictionary indexes and nested-data levels.
- Footer metadata exposes schemas, row groups, encodings, compression, statistics, sizes, and offsets.
- Page headers exist on disk, but PyArrow's public metadata API generally summarizes column chunks rather than exposing each page header directly.

Generated file: `C:\data\output\events_intro.parquet`
